In [1]:
# Install all dependency packages for the course.
# In notebooks, %pip installs into the active kernel environment.
%pip install --upgrade -r requirements.txt


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


## 2.3. Setup Function Tools for ReAct Agent

In [1]:
from langchain_core.tools import tool

#Tool annotation identifies a function as a tool automatically
@tool
def find_sum(x:int, y:int) -> int :
    #The docstring comment describes the capabilities of the function
    #It is used by the agent to discover the function's inputs, outputs and capabilities
    """
    This function is used to add two numbers and return their sum.
    It takes two integers as inputs and returns an integer as output.
    """
    return x + y

@tool
def find_product(x:int, y:int) -> int :
    """
    This function is used to multiply two numbers and return their product.
    It takes two integers as inputs and returns an integer as ouput.
    """
    return x * y


## 2.4. Create a basic ReAct Agent

In [ ]:
from getpass import getpass
from langchain_openai import ChatOpenAI
import os

# Setup the LLM for the agent. The Azure OpenAI API key is read from the
# environment when available, or requested interactively in the notebook.
if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass("AZURE_OPENAI_API_KEY: ")

azure_endpoint = os.getenv(
    "AZURE_OPENAI_ENDPOINT",
    "https://agentic-ai-course-kumaran.openai.azure.com/",
).rstrip("/")
azure_base_url = f"{azure_endpoint}/openai/v1/"

model = ChatOpenAI(
    model=os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o"),
    base_url=azure_base_url,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

# Test the model
# response = model.invoke("Hello, how are you?")
# print(response.content)


In [ ]:
from langchain.agents import create_agent

# Create list of tools available to the agent
agent_tools = [find_sum, find_product]

# System prompt
system_prompt = """You are a Math genius who can solve math problems. Solve the
problems provided by the user, by using only tools available.
Do not solve the problem yourself."""

agent_graph = create_agent(
    model=model,
    tools=agent_tools,
    system_prompt=system_prompt,
)


## 2.5. Execute the ReAct Agent

In [ ]:
#Example 1
inputs = {"messages":[("user","what is the sum of 2 and 3 ?")]}

result = agent_graph.invoke(inputs)

#Get the final answer
print(f"Agent returned : {result['messages'][-1].content} \n")

print("Step by Step execution : ")
for message in result['messages']:
    print(message.pretty_repr())

In [ ]:
#Example 2
inputs = {"messages":[("user","What is 3 multipled by 2 and 5 + 1 ?")]}

result = agent_graph.invoke(inputs)

#Get the final answer
print(f"Agent returned : {result['messages'][-1].content} \n")

print("Step by Step execution : ")
for message in result['messages']:
    print(message.pretty_repr())

## 2.6. Debugging the Agent

In [ ]:
agent_graph = create_agent(
    model=model,
    tools=agent_tools,
    system_prompt=system_prompt,
    debug=True,
)

inputs = {"messages": [("user", "what is the sum of 2 and 3 ?")]}

result = agent_graph.invoke(inputs)
